In [64]:
import pandas as pd


In [65]:
df = pd.read_csv('pizza_sales.csv')
df.head()

,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1,1,hawaiian_m,1,1/1/2015,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2,2,classic_dlx_m,1,1/1/2015,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3,2,five_cheese_l,1,1/1/2015,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4,2,ital_supr_l,1,1/1/2015,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5,2,mexicana_m,1,1/1/2015,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


In [66]:
df.info()
#info function used to check all column data types. Notably, fields order_date and order_time data type needs to be fixed. 


<class 'pandas.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pizza_id           48620 non-null  int64  
 1   order_id           48620 non-null  int64  
 2   pizza_name_id      48620 non-null  str    
 3   quantity           48620 non-null  int64  
 4   order_date         48620 non-null  str    
 5   order_time         48620 non-null  str    
 6   unit_price         48620 non-null  float64
 7   total_price        48620 non-null  float64
 8   pizza_size         48620 non-null  str    
 9   pizza_category     48620 non-null  str    
 10  pizza_ingredients  48620 non-null  str    
 11  pizza_name         48620 non-null  str    
dtypes: float64(2), int64(3), str(7)
memory usage: 10.4 MB


Converting all date time fields to type dateTime using a function for reusability

In [67]:
def converToDateTime(df,columns):
    for col in columns:
        df[col] = pd.to_datetime(df[col], format ='mixed')

converToDateTime(df, ['order_date'])

Conver to Currency

In [68]:
def currencyConverter(df, prefix, *args):
    for col in args:
        df[col] = df[col].map(f"{prefix}""{:,.2f}".format)

Checking for duplicates

In [69]:
df[df.duplicated(subset = 'pizza_id')].count()

pizza_id             0
order_id             0
pizza_name_id        0
quantity             0
order_date           0
order_time           0
unit_price           0
total_price          0
pizza_size           0
pizza_category       0
pizza_ingredients    0
pizza_name           0
dtype: int64

Check data date ranges

In [70]:
latest_date = df['order_date'].max()
earliest_date = df['order_date'].min()
print(f"{earliest_date} - {latest_date}")

2015-01-01 00:00:00 - 2015-12-31 00:00:00


FEATURE ENGINEERING

In [71]:
df['order_day_name']= df['order_date'].dt.day_name()
df['order_month_number'] = df['order_date'].dt.month
df['order_month_name'] = df['order_date'].dt.month_name()

In [72]:
'''
Key Performance Indicators (KPIs)
•	Total Revenue = Sum of total_price
•	Total Pizzas Sold = Sum of quantity
•	Total Orders = Count of unique order_id
•	Average Order Value (AOV) = Total Revenue ÷ Total Orders
•	Average Pizza per Order = Total Pizzas Sold ÷ Total Orders

'''

'\nKey Performance Indicators (KPIs)\n•\tTotal Revenue = Sum of total_price\n•\tTotal Pizzas Sold = Sum of quantity\n•\tTotal Orders = Count of unique order_id\n•\tAverage Order Value (AOV) = Total Revenue ÷ Total Orders\n•\tAverage Pizza per Order = Total Pizzas Sold ÷ Total Orders\n\n'

In [73]:
total_revenue = df['total_price'].sum()
total_pizzas_sold = df['pizza_id'].count()
total_orders = df['order_id'].nunique()
average_order_value = total_revenue / total_orders
average_pizza_per_order = total_pizzas_sold / total_orders

print(f"Total Revenue: {total_revenue}")
print(f"Total Pizzas Sold: {total_pizzas_sold}")
print(f"Total Orders: {total_orders}")
print(f"Average Order Value: {average_order_value}")
print(f"Average Pizzas per Order: {average_pizza_per_order}")

Total Revenue: 817860.0499999999
Total Pizzas Sold: 48620
Total Orders: 21350
Average Order Value: 38.30726229508196
Average Pizzas per Order: 2.2772833723653396


ANALYSIS

In [74]:
weekdays = ['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']
df['order_day_name'] = pd.Categorical(df['order_day_name'], categories=weekdays)

In [ ]:
def salesAnalysis(df,colname, )

In [75]:
#Sales by the day of the week
sales_by_day = df.groupby(['order_day_name'], observed=False).agg(
    Total_Revenue = ('total_price', 'sum'),
    Total_Pizzas_Sold = ('pizza_id','count'),
    Total_Orders = ('order_id', 'nunique'),
).reset_index().rename(
    columns = {
        'order_day_name': 'Order Day'
    }
)
sales_by_day['Average_Order_Value'] = sales_by_day['Total_Revenue'] /sales_by_day['Total_Orders']
sales_by_day['Average_Pizza_Per_Order'] = (sales_by_day['Total_Pizzas_Sold']/ sales_by_day['Total_Orders']).map("{:.2f}".format)

currencyConverter(sales_by_day, 'Rs. ','Total_Revenue', 'Average_Order_Value' )
sales_by_day.columns = sales_by_day.columns.str.replace('_',' ', regex=False)

sales_by_day

,Order Day,Total Revenue,Total Pizzas Sold,Total Orders,Average Order Value,Average Pizza Per Order
0,Sunday,"Rs. 102,116.45",6063,2710,Rs. 37.68,2.24
1,Monday,"Rs. 110,471.60",6577,2940,Rs. 37.58,2.24
2,Tuesday,"Rs. 115,594.45",6833,2978,Rs. 38.82,2.29
3,Wednesday,"Rs. 116,731.20",6907,3064,Rs. 38.10,2.25
4,Thursday,"Rs. 121,650.30",7243,3173,Rs. 38.34,2.28
5,Friday,"Rs. 129,690.90",7723,3359,Rs. 38.61,2.30
6,Saturday,"Rs. 121,605.15",7274,3126,Rs. 38.90,2.33


In [84]:
sales_by_month = df.groupby(['order_month_number','order_month_name']).agg(
    Total_Revenue = ('total_price', 'sum'),
    Total_Pizzas_Sold = ('pizza_id','count'),
    Total_Orders = ('order_id', 'nunique'),
).reset_index().rename(
    columns = {
        'order_month_name': 'Order Month'
    }    
)[['Order Month','Total_Revenue','Total_Pizzas_Sold','Total_Orders']]

sales_by_month['Average_Order_Value'] = sales_by_month['Total_Revenue'] /sales_by_month['Total_Orders']
sales_by_month['Average_Pizza_Per_Order'] = (sales_by_month['Total_Pizzas_Sold']/ sales_by_month['Total_Orders']).map("{:.2f}".format)


currencyConverter(sales_by_month, 'Rs. ','Total_Revenue', 'Average_Order_Value')
sales_by_month.columns = sales_by_month.columns.str.replace('_',' ',regex=False)

sales_by_month


,Order Month,Total Revenue,Total Pizzas Sold,Total Orders,Average Order Value,Average Pizza Per Order
0,January,"Rs. 71,620.15",4288,1929,Rs. 37.13,2.22
1,February,"Rs. 64,419.45",3821,1648,Rs. 39.09,2.32
2,March,"Rs. 71,301.40",4251,1864,Rs. 38.25,2.28
3,April,"Rs. 70,312.00",4182,1829,Rs. 38.44,2.29
4,May,"Rs. 67,648.80",4008,1765,Rs. 38.33,2.27
5,June,"Rs. 68,161.45",4044,1771,Rs. 38.49,2.28
6,July,"Rs. 70,880.65",4212,1860,Rs. 38.11,2.26
7,August,"Rs. 69,497.30",4141,1835,Rs. 37.87,2.26
8,September,"Rs. 63,803.70",3780,1638,Rs. 38.95,2.31
9,October,"Rs. 68,152.20",4045,1782,Rs. 38.24,2.27
